In [1]:
import pandas as pd
import numpy as np
import os

# Load processed signal data (daily raw + weekday raw)
DATA_FOLDER = "Signal_output" 
daily_raw_path = os.path.join(DATA_FOLDER, "signal_daily_raw.csv")
weekday_raw_path = os.path.join(DATA_FOLDER, "signal_weekday_avg_raw.csv")
meta_path = os.path.join(DATA_FOLDER, "signal_metadata.csv")

daily_df = pd.read_csv(daily_raw_path)
weekday_df = pd.read_csv(weekday_raw_path)
meta = pd.read_csv(meta_path)

print("Daily shape:", daily_df.shape)
print("Weekday shape:", weekday_df.shape)

Daily shape: (49454, 46)
Weekday shape: (1890, 47)


In [2]:
# Merge metadata into aggregated data
meta_cols_to_add = meta.drop(columns=["intersection_type"])

daily_df = daily_df.merge(meta_cols_to_add, on="SignalID", how="left")
weekday_df = weekday_df.merge(meta_cols_to_add, on="SignalID", how="left")

print("Daily shape:", daily_df.shape)
print("Weekday shape:", weekday_df.shape)

Daily shape: (49454, 51)
Weekday shape: (1890, 52)


In [3]:
# Compute turning ratios for each approach (L/T/R)
# Ratios are:
#   left_ratio   = left_total   / (left_total + through_total + right_total)
#   through_ratio= through_total/ ...
#   right_ratio  = right_total  / ...
print("daily_df columns:", daily_df.columns.tolist())
print("weekday_df columns:", weekday_df.columns.tolist())

def compute_overall_turn_ratios(df):
    df = df.copy()
    dir_prefix = {
        "east":  "Vehicle_Eastbound",
        "west":  "Vehicle_Westbound",
        "north": "Vehicle_Northbound",
        "south": "Vehicle_Southbound"
    }

    results = []

    # Compute per SignalID
    for sid, g in df.groupby("SignalID"):

        row = g.copy()

        # Determine usable directions for THIS signal
        valid_dirs = [
            d for d in dir_prefix.keys()
            if row[f"decomp_{d}"].iloc[0] == 1
        ]

        # Columns to sum
        left_cols    = [dir_prefix[d] + "_L" for d in valid_dirs]
        through_cols = [dir_prefix[d] + "_T" for d in valid_dirs]
        right_cols   = [dir_prefix[d] + "_R" for d in valid_dirs]

        # Only sum valid directions
        row["left_total"]    = row[left_cols].sum(axis=1) if left_cols else 0
        row["through_total"] = row[through_cols].sum(axis=1) if through_cols else 0
        row["right_total"]   = row[right_cols].sum(axis=1) if right_cols else 0

        row["turn_total"] = row["left_total"] + row["through_total"] + row["right_total"]

        row["left_ratio"] = np.where(row["turn_total"] > 0,
                                     row["left_total"] / row["turn_total"], np.nan)
        row["through_ratio"] = np.where(row["turn_total"] > 0,
                                        row["through_total"] / row["turn_total"], np.nan)
        row["right_ratio"] = np.where(row["turn_total"] > 0,
                                      row["right_total"] / row["turn_total"], np.nan)

        results.append(row)

    result_df = pd.concat(results, ignore_index=True)

    return result_df


daily_ratio = compute_overall_turn_ratios(daily_df)
weekday_ratio = compute_overall_turn_ratios(weekday_df)

# Drop signals with no usable movements (turn_total == 0)
daily_ratio = daily_ratio[daily_ratio["turn_total"] > 0].copy()
weekday_ratio = weekday_ratio[weekday_ratio["turn_total"] > 0].copy()

print("Remaining signals in daily_ratio:", daily_ratio["SignalID"].nunique())
print("Remaining signals in weekday_ratio:", weekday_ratio["SignalID"].nunique())

daily_df columns: ['SignalID', 'date', 'intersection_type', 'Vehicle_Eastbound_TL', 'Vehicle_Eastbound_T', 'Vehicle_Eastbound_Total', 'Vehicle_Westbound_TL', 'Vehicle_Westbound_T', 'Vehicle_Westbound_Total', 'Vehicle_Northbound_TL', 'Vehicle_Northbound_Total', 'Vehicle_Southbound_TL', 'Vehicle_Southbound_Total', 'Vehicle_Vehicle_Total', 'Vehicle_Northbound_L', 'Vehicle_Northbound_TR', 'Vehicle_Southbound_L', 'Vehicle_Southbound_TR', 'Vehicle_Eastbound_L', 'Vehicle_Eastbound_R', 'Vehicle_Northbound_T', 'Vehicle_Southbound_T', 'Vehicle_Westbound_L', 'Vehicle_Westbound_R', 'Vehicle_Westbound_TR', 'Vehicle_Eastbound_TR', 'Vehicle_Southbound_R', 'Exit_Northbound_T', 'Exit_Northbound_Total', 'Exit_Southbound_T', 'Exit_Southbound_Total', 'Exit_Exit_Total', 'Vehicle_Northbound_R', 'Vehicle_Northeast_TL', 'Vehicle_Northeast_Total', 'Exit_Eastbound_T', 'Exit_Eastbound_Total', 'Exit_Westbound_T', 'Exit_Westbound_Total', 'Exit_Eastbound_L', 'Vehicle_Northwest_L', 'Vehicle_Northwest_Total', 'Exit_E

In [4]:
# Station-level overall L/T/R ratios
station_totals = (
    daily_ratio.groupby("SignalID", as_index=False)[
        ["left_total", "through_total", "right_total"]
    ].sum()
)

station_totals["turn_total"] = (
    station_totals["left_total"] +
    station_totals["through_total"] +
    station_totals["right_total"]
)

station_totals["left_ratio"] = station_totals["left_total"] / station_totals["turn_total"]
station_totals["through_ratio"] = station_totals["through_total"] / station_totals["turn_total"]
station_totals["right_ratio"] = station_totals["right_total"] / station_totals["turn_total"]

print("\nStation-level L/T/R ratios:")
print(station_totals[["SignalID", "left_ratio", "through_ratio", "right_ratio"]].head())

# Save for inspection
station_ratio_path = os.path.join(DATA_FOLDER, "signal_station_ratio_check.csv")
station_totals.to_csv(station_ratio_path, index=False)
print("Saved station-level ratio check to:", station_ratio_path)



Station-level L/T/R ratios:
   SignalID  left_ratio  through_ratio  right_ratio
0       178    0.246731       0.374605     0.378664
1       195    0.000000       1.000000     0.000000
2       196    1.000000       0.000000     0.000000
3       391    0.000000       1.000000     0.000000
4       475    0.559393       0.440607     0.000000
Saved station-level ratio check to: Signal_output/signal_station_ratio_check.csv


In [5]:
# Preview daily turn ratios
print(daily_ratio.columns.tolist())
daily_ratio[[
    "SignalID", "date",
    "intersection_type",
    "left_ratio", "through_ratio", "right_ratio"
]].head()

['SignalID', 'date', 'intersection_type', 'Vehicle_Eastbound_TL', 'Vehicle_Eastbound_T', 'Vehicle_Eastbound_Total', 'Vehicle_Westbound_TL', 'Vehicle_Westbound_T', 'Vehicle_Westbound_Total', 'Vehicle_Northbound_TL', 'Vehicle_Northbound_Total', 'Vehicle_Southbound_TL', 'Vehicle_Southbound_Total', 'Vehicle_Vehicle_Total', 'Vehicle_Northbound_L', 'Vehicle_Northbound_TR', 'Vehicle_Southbound_L', 'Vehicle_Southbound_TR', 'Vehicle_Eastbound_L', 'Vehicle_Eastbound_R', 'Vehicle_Northbound_T', 'Vehicle_Southbound_T', 'Vehicle_Westbound_L', 'Vehicle_Westbound_R', 'Vehicle_Westbound_TR', 'Vehicle_Eastbound_TR', 'Vehicle_Southbound_R', 'Exit_Northbound_T', 'Exit_Northbound_Total', 'Exit_Southbound_T', 'Exit_Southbound_Total', 'Exit_Exit_Total', 'Vehicle_Northbound_R', 'Vehicle_Northeast_TL', 'Vehicle_Northeast_Total', 'Exit_Eastbound_T', 'Exit_Eastbound_Total', 'Exit_Westbound_T', 'Exit_Westbound_Total', 'Exit_Eastbound_L', 'Vehicle_Northwest_L', 'Vehicle_Northwest_Total', 'Exit_Eastbound_R', 'Exit

,SignalID,date,intersection_type,left_ratio,through_ratio,right_ratio
370,178,2025-05-01,T_intersection,0.245509,0.392446,0.362045
371,178,2025-05-02,T_intersection,0.242879,0.360731,0.396391
372,178,2025-05-03,T_intersection,0.252747,0.368874,0.378378
373,178,2025-05-04,T_intersection,0.162205,0.435039,0.402756
374,178,2025-05-05,T_intersection,0.252113,0.369249,0.378638


In [6]:
# Preview weekday-average turn ratios
weekday_ratio[[
    "SignalID", "dow", "dow_name",
    "intersection_type",
    "left_ratio", "through_ratio", "right_ratio"
]].head()

,SignalID,dow,dow_name,intersection_type,left_ratio,through_ratio,right_ratio
14,178,0.0,Monday,T_intersection,0.275535,0.364651,0.359814
15,178,1.0,Tuesday,T_intersection,0.273014,0.378051,0.348935
16,178,2.0,Wednesday,T_intersection,0.269708,0.378044,0.352249
17,178,3.0,Thursday,T_intersection,0.264287,0.374557,0.361156
18,178,4.0,Friday,T_intersection,0.247358,0.369965,0.382677


In [7]:
# Daily L/T/R ratios by intersection type (across all signals)
ratio_cols = ["left_ratio", "through_ratio", "right_ratio"]

daily_type_daily = (
    daily_ratio
    .groupby(["intersection_type", "date"])[ratio_cols]
    .mean()
    .reset_index()
)

daily_type_daily.head()

,intersection_type,date,left_ratio,through_ratio,right_ratio
0,2_way,2025-05-01,0.412086,0.587914,0.0
1,2_way,2025-05-02,0.410259,0.589741,0.0
2,2_way,2025-05-03,0.414133,0.585867,0.0
3,2_way,2025-05-04,0.412498,0.587502,0.0
4,2_way,2025-05-05,0.409886,0.590114,0.0


In [8]:
# Weekday L/T/R ratios by intersection type (across all signals)
weekday_type_daily = (
    weekday_ratio
    .groupby(["intersection_type", "dow", "dow_name"])[ratio_cols]
    .mean()
    .reset_index()
)

weekday_type_daily.head()

,intersection_type,dow,dow_name,left_ratio,through_ratio,right_ratio
0,2_way,0.0,Monday,0.411843,0.588157,0.0
1,2_way,1.0,Tuesday,0.411470,0.588530,0.0
2,2_way,2.0,Wednesday,0.411648,0.588352,0.0
3,2_way,3.0,Thursday,0.411824,0.588176,0.0
4,2_way,4.0,Friday,0.411675,0.588325,0.0


In [9]:
# Overall L/T/R ratios by intersection type
overall_type_summary = (
    daily_ratio
    .groupby("intersection_type")[ratio_cols]
    .mean()
    .reset_index()
)

overall_type_summary

,intersection_type,left_ratio,through_ratio,right_ratio
0,2_way,0.413770,0.586230,0.00000
1,4_way,0.140137,0.846643,0.01322
2,T_intersection,0.179257,0.778994,0.04175
3,other,1.000000,0.000000,0.00000


In [10]:
output_daily_by_date = os.path.join(DATA_FOLDER, "intersection_turn_ratios_daily_by_date.csv")
daily_type_daily.to_csv(output_daily_by_date, index=False)
print("Saved:", output_daily_by_date)

output_weekday_by_dow = os.path.join(DATA_FOLDER, "intersection_turn_ratios_weekday_by_dow.csv")
weekday_type_daily.to_csv(output_weekday_by_dow, index=False)
print("Saved:", output_weekday_by_dow)

output_overall = os.path.join(DATA_FOLDER, "intersection_turn_ratios_overall.csv")
overall_type_summary.to_csv(output_overall, index=False)
print("Saved:", output_overall)


Saved: Signal_output/intersection_turn_ratios_daily_by_date.csv
Saved: Signal_output/intersection_turn_ratios_weekday_by_dow.csv
Saved: Signal_output/intersection_turn_ratios_overall.csv
